# download and process datasets used in this paper

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import scvi
import os
import anndata as ad
from pathlib import Path


# Stephenson

In [37]:
# ref https://www.ebi.ac.uk/biostudies/arrayexpress/studies/E-MTAB-10026
!wget -O covid_portal_210320_with_raw.h5ad 'https://ftp.ebi.ac.uk/biostudies/fire/E-MTAB-/026/E-MTAB-10026/Files/covid_portal_210320_with_raw.h5ad'

--2026-07-08 05:41:37--  https://ftp.ebi.ac.uk/biostudies/fire/E-MTAB-/026/E-MTAB-10026/Files/covid_portal_210320_with_raw.h5ad
Resolving ftp.ebi.ac.uk (ftp.ebi.ac.uk)... 193.62.193.165
Connecting to ftp.ebi.ac.uk (ftp.ebi.ac.uk)|193.62.193.165|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7187322881 (6.7G)
Saving to: ‘covid_portal_210320_with_raw.h5ad’

covid_portal_210320 100%[===================>]   6.69G  17.6MB/s    in 8m 30s  

2026-07-08 05:50:08 (13.4 MB/s) - ‘covid_portal_210320_with_raw.h5ad’ saved [7187322881/7187322881]



In [ ]:
adata = sc.read_h5ad("covid_portal_210320_with_raw.h5ad")
adata = adata[adata.obs['Status']=='Healthy']

adata_gex = sc.AnnData(X = adata.layers['raw'][:,:24737],
    obs = adata.obs_names.to_numpy(),
    var = adata.var['feature_types'].to_frame().index.to_numpy()[:24737])
adata_gex.obs_names = adata.obs_names
adata_gex.var_names = adata.var_names[:24737]
adata_gex.layers['counts'] = adata_gex.X.copy()
adata_gex.var.columns = ['Features']
adata_gex.obs.columns = ['barcode']
adata_gex.obs['batch'] = np.array(adata.obs['patient_id'].copy())
adata_gex.obs['cell_type'] = adata.obs['full_clustering'].to_numpy().copy()



adata_adt = sc.AnnData(X = adata.layers['raw'][:,24737:],
    obs = adata.obs_names.to_numpy(),
    var = adata.var['feature_types'].to_frame().index.to_numpy()[24737:])
adata_adt.obs_names = adata.obs_names
adata_adt.var_names = adata.var_names[24737:]
adata_adt.var.columns = ['Features']
adata_adt.obs.columns = ['barcode']
adata_adt.obs['batch'] = np.array(adata.obs['patient_id'].copy())
adata_adt.obs['cell_type'] = adata.obs['full_clustering'].to_numpy().copy()

adata_gex.obsm['protein_counts'] = pd.DataFrame(adata_adt.X.toarray().copy(), index = adata_adt.obs_names, columns = adata_adt.var_names)
adata_gex.var_names_make_unique()
adata_gex.write("RNA_ADT/Stephenson/haniffa21.processed_healthy.h5ad")

# neurIPS data

In [14]:
# ref https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE194122
!wget  -O GSE194122_openproblems_neurips2021_cite_BMMC_processed.h5ad.gz 'https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE194122&format=file&file=GSE194122%5Fopenproblems%5Fneurips2021%5Fcite%5FBMMC%5Fprocessed%2Eh5ad%2Egz'
!gzip -d GSE194122_openproblems_neurips2021_cite_BMMC_processed.h5ad.gz

--2026-07-08 05:26:04--  https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE194122&format=file&file=GSE194122%5Fopenproblems%5Fneurips2021%5Fcite%5FBMMC%5Fprocessed%2Eh5ad%2Egz
Resolving www.ncbi.nlm.nih.gov (www.ncbi.nlm.nih.gov)... 130.14.29.110, 2607:f220:41e:4290::110
Connecting to www.ncbi.nlm.nih.gov (www.ncbi.nlm.nih.gov)|130.14.29.110|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 615842052 (587M) [application/octet-stream]
Saving to: ‘GSE194122_openproblems_neurips2021_cite_BMMC_processed.h5ad.gz’

GSE194122_openprobl 100%[===================>] 587.31M  12.3MB/s    in 35s     

2026-07-08 05:26:39 (16.8 MB/s) - ‘GSE194122_openproblems_neurips2021_cite_BMMC_processed.h5ad.gz’ saved [615842052/615842052]



In [ ]:
adata = sc.read_h5ad("GSE194122_openproblems_neurips2021_cite_BMMC_processed.h5ad")
adata_gex = sc.AnnData(X = adata.layers['counts'][:,:13953].copy())
adata_gex.obs_names = adata.obs_names.copy()
adata_gex.var_names = adata.var_names[:13953].copy()
adata_gex.obs['cell_type'] = adata.obs['cell_type'].copy()
adata_gex.obs['batch'] = adata.obs['batch'].copy()
adata_gex.obsm['protein_counts'] = adata.layers['counts'][:,13953:].copy().toarray()
adata_gex.layers['counts'] = adata_gex.X.copy()
adata_gex.var_names_make_unique()
adata_gex.obsm['protein_counts'] = pd.DataFrame(adata_gex.obsm['protein_counts'].copy(), adata_gex.obs_names, adata.var_names[13953:].copy())
adata_gex.write("RNA_ADT/neurIPS/GSE194122_openproblems_neurips2021_cite_BMMC_processed.h5ad")

In [36]:
adata_gex

AnnData object with n_obs × n_vars = 90261 × 13953
    obs: 'cell_type', 'batch'
    obsm: 'protein_counts'
    layers: 'counts'

# Hao

In [ ]:
scvi.data.pbmc_seurat_v4_cite_seq()
adata = sc.read_h5ad("data/pbmc_seurat_v4.h5ad")
adata.layers['counts'] = adata_gex.X.copy()
adata.write("RNA_ADT/Hao/pbmc_seurat_v4.h5ad")
adata

INFO     File data/pbmc_seurat_v4.h5ad already downloaded                                                          


# BMMC

In [1]:
# ref https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE194122
!wget -c -O GSE194122_openproblems_neurips2021_multiome_BMMC_processed.h5ad.gz "https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE194122&format=file&file=GSE194122%5Fopenproblems%5Fneurips2021%5Fmultiome%5FBMMC%5Fprocessed%2Eh5ad%2Egz"
!gzip -d GSE194122_openproblems_neurips2021_multiome_BMMC_processed.h5ad.gz

--2026-07-08 07:37:11--  https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE194122&format=file&file=GSE194122%5Fopenproblems%5Fneurips2021%5Fmultiome%5FBMMC%5Fprocessed%2Eh5ad%2Egz
Resolving www.ncbi.nlm.nih.gov (www.ncbi.nlm.nih.gov)... 130.14.29.110, 2607:f220:41e:4290::110
Connecting to www.ncbi.nlm.nih.gov (www.ncbi.nlm.nih.gov)|130.14.29.110|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2917117242 (2.7G) [application/octet-stream]
Saving to: ‘GSE194122_openproblems_neurips2021_multiome_BMMC_processed.h5ad.gz’

GSE194122_openprobl 100%[===================>]   2.72G  38.4MB/s    in 2m 36s  

2026-07-08 07:39:48 (17.8 MB/s) - ‘GSE194122_openproblems_neurips2021_multiome_BMMC_processed.h5ad.gz’ saved [2917117242/2917117242]



In [ ]:
adata = sc.read_h5ad("GSE194122_openproblems_neurips2021_multiome_BMMC_processed.h5ad")
adata.var['modality'] = adata.var['feature_types'].astype(str)
adata.var['modality'][adata.var['feature_types'] == "GEX"] = "Gene Expression"
adata.var['modality'][adata.var['feature_types'] == "ATAC"] = "Peaks"
adata.obs = adata.obs[['cell_type', 'batch', 'GEX_pseudotime_order', 'Samplename', 'Site', 'DonorNumber', 'Modality',  'DonorID', 'DonorAge', 'DonorGender']]
adata.var = adata.var[['gene_id', 'modality']]
adata.X = adata.layers['counts']
adata.write("RNA_ATAC/BMMC/BMMC.h5ad")
adata

/tmp/ipykernel_34788/51362728.py:3: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

Try using '.loc[row_indexer, col_indexer] = value' instead, to perform the assignment in a single step.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html#chained-assignment
  adata.var['modality'][adata.var['feature_types'] == "GEX"] = "Gene Expression"
/tmp/ipykernel_34788/51362728.py:4: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-

AnnData object with n_obs × n_vars = 69249 × 129921
    obs: 'cell_type', 'batch', 'GEX_pseudotime_order', 'Samplename', 'Site', 'DonorNumber', 'Modality', 'DonorID', 'DonorAge', 'DonorGender'
    var: 'gene_id', 'modality'
    uns: 'ATAC_gene_activity_var_names', 'dataset_id', 'genome', 'organism'
    obsm: 'ATAC_gene_activity', 'ATAC_lsi_full', 'ATAC_lsi_red', 'ATAC_umap', 'GEX_X_pca', 'GEX_X_umap'
    layers: 'counts'

# Brain-ISSAAC

In [ ]:
# from multiomics benchmarking paper https://doi.org/10.1038/s41592-024-02429-w
# https://mailustceducn-my.sharepoint.com/personal/hyl2016_mail_ustc_edu_cn/_layouts/15/onedrive.aspx?id=%2Fpersonal%2Fhyl2016%5Fmail%5Fustc%5Fedu%5Fcn%2FDocuments%2FMultiomicsBenchmark%2FRaw%5Fdata%2FRNA%5FATAC%2F4%5Fbrain%5FISSAAC%5Fseq&viewid=b6d1a33b%2D630a%2D4d98%2Db85e%2D2df575b1c642
# original data https://www.ebi.ac.uk/biostudies/arrayexpress/studies/E-MTAB-11264
rna = sc.read_mtx("RNA/matrix.mtx").T
rna.var_names = pd.read_csv("RNA/features.tsv", sep="\t", header=None)[0].values
rna.obs_names = pd.read_csv("RNA/barcodes.tsv", sep="\t", header=None)[0].values
rna.var['modality'] = 'Gene Expression'

atac = sc.read_mtx("ATAC/matrix.mtx").T
atac.var_names = pd.read_csv("ATAC/features.tsv", sep="\t", header=None)[0].values
atac.obs_names = pd.read_csv("ATAC/barcodes.tsv", sep="\t", header=None)[0].values
atac.var['modality'] = 'Peaks'

combined = sc.concat([rna, atac], axis=1, join="inner")
metadata = pd.read_csv("meta_data.csv", index_col=0)

combined.obs["cell_type"] = metadata['celltype']
combined.obs['batch'] = 1
combined.write_h5ad("RNA_ATAC/Brain-ISSAAC-seq.h5ad")
combined

# TEA-seq

In [ ]:
# from multiomics benchmarking paper https://doi.org/10.1038/s41592-024-02429-w
# https://mailustceducn-my.sharepoint.com/personal/hyl2016_mail_ustc_edu_cn/_layouts/15/onedrive.aspx?id=%2Fpersonal%2Fhyl2016%5Fmail%5Fustc%5Fedu%5Fcn%2FDocuments%2FMultiomicsBenchmark%2FRaw%5Fdata%2FRNA%5FProtein%2F20%5FGSE158013&viewid=b6d1a33b%2D630a%2D4d98%2Db85e%2D2df575b1c642
# original data https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE158013
rna = sc.read_mtx("./RNA/matrix.mtx").T
rna.var_names = pd.read_csv("./RNA/features.tsv", sep="\t", header=None)[0].values
rna.obs_names = pd.read_csv("./RNA/barcodes.tsv", sep="\t", header=None)[0].values
rna.var['modality'] = 'Gene Expression'

atac = sc.read_mtx("./ATAC/matrix.mtx").T
atac.var_names = pd.read_csv("ATAC/features.tsv", sep="\t", header=None)[0].values
atac.obs_names = pd.read_csv("ATAC/barcodes.tsv", sep="\t", header=None)[0].values
atac.var['modality'] = 'Peaks'

adt = pd.read_csv("./ADT.csv", index_col=0).T
adt.index = adt.index.str.replace(".", "-")

combined = sc.concat([rna, atac], axis=1, join="inner")
combined.obsm['protein_expression'] = adt

metadata = pd.read_csv("metadata.csv", index_col=0)
combined.obs["cell_type"] = metadata['celltype']
combined.obs['batch'] = pd.Categorical(pd.Series(rna.obs_names).apply(lambda x: x.split("-")[1]))
# combined.write_h5ad("TEA-seq.h5ad")

# Breast Cancer

In [ ]:
# https://singlecell.broadinstitute.org/single_cell/data/public/SCP1039/a-single-cell-and-spatially-resolved-atlas-of-human-breast-cancers?filename=CITE.zip
!wget -O CITE.zip 'https://singlecell.broadinstitute.org/single_cell/data/public/SCP1039/a-single-cell-and-spatially-resolved-atlas-of-human-breast-cancers?filename=CITE.zip'
!gzip -d CITE.zip

In [ ]:
!wget -O GSE176078.tar 'https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE176078&format=file'
!gzip -d GSE176078.tar

In [ ]:
protein_root = Path("CITE")
gene_root = Path("GSE176078")
output_path = Path("breast_cancer_cite.h5ad")


def normalize_barcode(barcode: str) -> str:
    return barcode.split("_", 1)[1] if "_" in barcode else barcode


def load_protein_sample(sample_dir: Path) -> ad.AnnData:
    adata = sc.read_mtx(sample_dir / "umi_count" / "matrix.mtx").transpose()
    adata.var_names = pd.read_csv(sample_dir / "umi_count" / "features.tsv", header=None)[0].astype(str).to_numpy()
    adata.obs_names = pd.read_csv(sample_dir / "umi_count" / "barcodes.tsv", header=None)[0].astype(str).to_numpy()
    return adata


def load_gene_sample(sample_dir: Path) -> ad.AnnData:
    adata = sc.read_mtx(sample_dir / "count_matrix_sparse.mtx").transpose()
    adata.var_names = pd.read_csv(sample_dir / "count_matrix_genes.tsv", header=None)[0].astype(str).to_numpy()
    adata.obs_names = (
        pd.read_csv(sample_dir / "count_matrix_barcodes.tsv", header=None)[0]
        .astype(str)
        .map(normalize_barcode)
        .to_numpy()
    )

    metadata = pd.read_csv(sample_dir / "metadata.csv")
    adata.obs["celltype_minor"] = metadata["celltype_minor"].to_numpy()
    adata.obs["celltype_major"] = metadata["celltype_major"].to_numpy()
    return adata

In [ ]:
protein_sample_dirs = {
    path.name.split("_", 1)[0]: path
    for path in protein_root.iterdir()
    if path.is_dir()
}
gene_sample_dirs = {path.name[3:]: path for path in gene_root.iterdir() if path.is_dir()}
sample_ids = sorted(set(protein_sample_dirs) & set(gene_sample_dirs))

if not sample_ids:
    raise ValueError("No matching CITE and gene samples were found.")

paired_adatas = []
protein_var_names = None

for sample_id in sample_ids:
    protein_adata = load_protein_sample(protein_sample_dirs[sample_id])
    gene_adata = load_gene_sample(gene_sample_dirs[sample_id])

    shared_barcodes = protein_adata.obs_names.intersection(gene_adata.obs_names)
    protein_adata = protein_adata[shared_barcodes].copy()
    gene_adata = gene_adata[shared_barcodes].copy()

    if protein_var_names is None:
        protein_var_names = protein_adata.var_names.to_numpy()
    elif not np.array_equal(protein_var_names, protein_adata.var_names.to_numpy()):
        raise ValueError(f"Protein features do not match for sample {sample_id}.")

    gene_adata.obsm["protein_counts"] = protein_adata.X.copy()
    paired_adatas.append(gene_adata)

    print(f"{sample_id}: {gene_adata.n_obs} cells, {gene_adata.n_vars} genes, {protein_adata.n_vars} proteins")

if protein_var_names is None:
    raise RuntimeError("Protein feature names were not loaded.")

adata = ad.concat(paired_adatas, join="outer", label="batch", keys=sample_ids, index_unique="-")
adata.uns["protein_var_names"] = protein_var_names
adata.obsm['protein_counts'] = pd.DataFrame(adata.obsm['protein_counts'].todense(), index=adata.obs_names, columns=adata.uns['protein_var_names'])
print(adata)

In [ ]:
adata.write(output_path)
print(f"Saved to {output_path}")

# Thymocyte dataset

In [ ]:
# ref https://cellxgene.cziscience.com/collections/7e216a15-82df-46ee-b454-d0261d99e5f5
# ref https://github.com/YosefLab/Thymus_CITE-seq/tree/main
!wget https://datasets.cellxgene.cziscience.com/f313847f-3318-4f32-8769-d280e89541a5.h5ad